# FIUBench PO Forget5 — 단일 T4, batch 4, accumulation 32

지난번 성공 PO와 같은 **fast tokenizer 계열**을 사용하면서, 요청한 batch 4를 T4에서 실행하기 위해 DeepSpeed ZeRO-2 CPU offload를 유지하는 실험입니다.

- forget microbatch: `4`
- GPU 프로세스: `1`
- gradient accumulation: `32`
- effective forget batch: `4 × 1 × 32 = 128`
- PO는 매 microbatch에 retain 4개도 함께 사용하므로 optimizer update당 총 256개 sequence를 처리합니다.
- 8 epochs, epoch당 4 optimizer updates, 총 32 updates
- fast-tokenizer 호환 환경: Transformers 4.46.3, tokenizers 0.20.3, PEFT 0.13.2

Kaggle에서 GPU를 켜고 Internet을 On으로 설정하세요. T4가 두 장 보이더라도 학습 명령은 `CUDA_VISIBLE_DEVICES=0`으로 첫 번째 GPU만 사용합니다. 설치 뒤 커널 재시작이 필요하므로 **Run All을 누르지 말고 셀을 순서대로 실행**하세요. batch 4는 T4 메모리에 빠듯할 수 있어 후속 PO와 같은 CPU offload를 유지합니다.


In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working
if [ ! -d FIUBench_kaggle/.git ]; then
  git clone https://github.com/npngjjm/FIUBench_kaggle.git
fi
cd FIUBench_kaggle
git checkout --detach 5463c79838c21b57e8a9540b3a99dc453b0c4ee2
git rev-parse HEAD
nvidia-smi
apt-get update -qq
apt-get install -y libaio-dev
python -m pip uninstall -y torchao
python -m pip install 'transformers==4.46.3' 'tokenizers==0.20.3' 'peft==0.13.2' 'accelerate==1.14.0' 'deepspeed==0.19.7' 'huggingface_hub==0.36.2' 'numpy==2.1.3' hydra-core omegaconf datasets sentencepiece protobuf scipy ninja pillow pyyaml requests tqdm


## 커널 재시작

위 설치 셀이 끝나면 아래 셀을 한 번 실행하세요. 세션이 끊긴 것이 아니라 Python 커널만 재시작됩니다. 다시 연결된 뒤 그 다음 셀부터 계속 실행하세요.


In [ ]:
import os, signal
os.kill(os.getpid(), signal.SIGKILL)


In [ ]:
import os, sys, json, subprocess
from pathlib import Path
from importlib.metadata import version
import torch, transformers, tokenizers, peft, accelerate, deepspeed

repo = Path('/kaggle/working/FIUBench_kaggle')
assert repo.is_dir(), '저장소가 없습니다. 설치 셀부터 실행하세요.'
os.chdir(repo)
os.environ['HF_HOME'] = '/kaggle/working/hf_cache'

expected = {
    'transformers': '4.46.3',
    'tokenizers': '0.20.3',
    'peft': '0.13.2',
    'accelerate': '1.14.0',
    'deepspeed': '0.19.7',
    'huggingface-hub': '0.36.2',
    'numpy': '2.1.3',
}
for name, wanted in expected.items():
    actual = version(name)
    print(name, actual)
    assert actual == wanted, (name, actual, wanted)

print('python:', sys.version)
print('torch:', torch.__version__, 'cuda:', torch.version.cuda)
assert torch.cuda.is_available()
assert torch.cuda.device_count() >= 1
assert 'T4' in torch.cuda.get_device_name(0), torch.cuda.get_device_name(0)
print('visible GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print('학습 시 CUDA_VISIBLE_DEVICES=0으로 GPU 한 장만 사용합니다.')
print('핵심 환경 확인 완료')


In [ ]:
# 저장소의 JSON과 prompt는 유지하고 공식 FIUBench 데이터셋에서 SFHQ.zip만 받습니다.
import json, shutil, zipfile
from collections import defaultdict
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download
from PIL import Image

repo = Path('/kaggle/working/FIUBench_kaggle')
for name in ['full.json', 'split.json', 'prompt.json']:
    assert (repo / 'dataset' / name).is_file(), name
records = [json.loads(s) for s in (repo / 'dataset/full.json').read_text().splitlines() if s.strip()]
data_revision = HfApi().dataset_info('gray311/FIUBench').sha
archive = hf_hub_download(
    'gray311/FIUBench', 'SFHQ.zip', repo_type='dataset', revision=data_revision,
    local_dir='/kaggle/working/fiu_download',
)

with zipfile.ZipFile(archive) as z:
    entries = defaultdict(list)
    for member in z.infolist():
        p = Path(member.filename)
        if not member.is_dir() and '__MACOSX' not in p.parts and not p.name.startswith('._'):
            entries[p.name].append(member)
    for record in records[:400]:
        target = (repo / record['image_path']).resolve()
        assert target.is_relative_to((repo / 'dataset').resolve())
        matches = entries[target.name]
        assert len(matches) == 1, (target.name, len(matches))
        target.parent.mkdir(parents=True, exist_ok=True)
        with z.open(matches[0]) as src, target.open('wb') as dst:
            shutil.copyfileobj(src, dst)
        with Image.open(target) as im:
            im.verify()

missing = [r['image_path'] for r in records[:400] if not (repo / r['image_path']).is_file()]
assert len(records[:400]) == 400 and not missing
Path('/kaggle/working/fiu_data_revision.txt').write_text(data_revision)
print('학습 대상 400개 이미지 검사 통과:', data_revision)


In [ ]:
from pathlib import Path
import yaml
from huggingface_hub import HfApi

repo = Path('/kaggle/working/FIUBench_kaggle')
train_config = repo / 'config/forget_po_forget5_single_t4_b4a32_fast.yaml'
accelerate_config = repo / 'config/accelerate_kaggle_single_t4_po.yaml'

train_config.write_text('''defaults:
  - forget_lora
  - _self_

model_family: llava-phi
model_path: gray311/vlm_unlearning_ft_llava_phi_3_mini
attn_implementation: sdpa
data_path: ./dataset/full.json
split: forget5
forget_loss: idk
lr: 3e-4
batch_size: 4
gradient_accumulation_steps: 32
num_epochs: 8
LoRA:
  r: 128
  alpha: 256
  dropout: 0.05
workers: 0
report_to: null
save_dir: /kaggle/working/models/llava_phi_PO_forget5_single_t4_b4a32_fast
save_model: true
save_steps: 8
''')

accelerate_config.write_text('''compute_environment: LOCAL_MACHINE
debug: false
deepspeed_config:
  gradient_accumulation_steps: 32
  gradient_clipping: 1.0
  offload_optimizer_device: cpu
  offload_param_device: cpu
  zero3_init_flag: false
  zero_stage: 2
distributed_type: DEEPSPEED
downcast_bf16: 'no'
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 1
rdzv_backend: static
same_network: true
tpu_env: []
tpu_use_cluster: false
tpu_use_sudo: false
use_cpu: false
''')

print(train_config.read_text())
print(accelerate_config.read_text())


In [ ]:
# 성공 PO 계열의 fast tokenizer와 gradient-checkpointing 호환 패치입니다.
import ast
from pathlib import Path

path = Path('/kaggle/working/FIUBench_kaggle/forget.py')
text = path.read_text()
original = text
old = 'AutoTokenizer.from_pretrained(cfg.model_path)'
old_slow = 'AutoTokenizer.from_pretrained(cfg.model_path, use_fast=False)'
new = 'AutoTokenizer.from_pretrained(cfg.model_path, use_fast=True)'
assert old in text or old_slow in text or new in text
if old_slow in text:
    text = text.replace(old_slow, new)
elif new not in text:
    text = text.replace(old, new)

marker = '    if cfg.LoRA.r != 0:\n        config = LoraConfig('
block = '''    if str(model_cfg.get("gradient_checkpointing", "false")).lower() == "true":
        model.config.use_cache = False
        model.gradient_checkpointing_enable()
        model.enable_input_require_grads()
        logger.info("Gradient checkpointing enabled", main_process_only=True)

'''
if 'model.gradient_checkpointing_enable()' not in text:
    assert text.count(marker) == 1
    text = text.replace(marker, block + marker, 1)

end = text.rfind('\nif __name__ == "__main__":')
assert end > 0
if 'if cfg.save_model and accelerator.is_main_process:' not in text:
    start = text.rfind('\n    accelerator.end_training()', 0, end)
    assert start > 0
    text = text[:start] + '''
    output_dir = cfg.save_dir
    accelerator.wait_for_everyone()
    if cfg.save_model and accelerator.is_main_process:
        os.makedirs(output_dir, exist_ok=True)
        unwrapped_model = accelerator.unwrap_model(model)
        if cfg.LoRA.r != 0:
            save_lora_weights(unwrapped_model, output_dir)
        else:
            unwrapped_model.save_pretrained(output_dir)
            tokenizer.save_pretrained(output_dir)
    accelerator.wait_for_everyone()
    accelerator.end_training()
''' + text[end:]

ast.parse(text)
backup = path.with_suffix('.py.before_single_t4_patch')
if not backup.exists():
    backup.write_text(original)
path.write_text(text)
assert 'use_fast=True' in text
assert 'model.enable_input_require_grads()' in text
print('fast tokenizer 런타임 패치 및 Python 구문 검사 완료')


In [ ]:
import json, os, subprocess
from pathlib import Path
from hydra import compose, initialize_config_dir
import yaml

repo = Path('/kaggle/working/FIUBench_kaggle')
os.chdir(repo)
with initialize_config_dir(version_base=None, config_dir=str(repo / 'config')):
    cfg = compose(config_name='forget_po_forget5_single_t4_b4a32_fast')
launcher = yaml.safe_load((repo / 'config/accelerate_kaggle_single_t4_po.yaml').read_text())

assert cfg.batch_size == 4
assert cfg.gradient_accumulation_steps == 32
assert launcher['num_processes'] == 1
assert launcher['deepspeed_config']['gradient_accumulation_steps'] == 32
assert cfg.batch_size * launcher['num_processes'] * cfg.gradient_accumulation_steps == 128
assert cfg.forget_loss == 'idk' and cfg.split == 'forget5'
assert cfg.num_epochs == 8 and cfg.save_steps == 8
assert cfg.attn_implementation == 'sdpa'
assert launcher['distributed_type'] == 'DEEPSPEED'
assert launcher['mixed_precision'] == 'fp16'
assert not (Path(cfg.save_dir) / 'checkpoint.pt').exists(), '기존 결과가 있습니다. 새 세션을 쓰거나 save_dir을 바꾸세요.'

source_revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
assert source_revision == '5463c79838c21b57e8a9540b3a99dc453b0c4ee2'
model_revision = HfApi().model_info(str(cfg.model_path)).sha
Path('/kaggle/working/fiu_environment_single_t4_b4a32_fast.txt').write_bytes(
    subprocess.check_output(['python', '-m', 'pip', 'freeze'])
)
Path('/kaggle/working/fiu_runtime_single_t4_b4a32_fast.patch').write_bytes(
    subprocess.check_output(['git', 'diff', '--', 'forget.py'])
)
manifest = {
    'source_revision': source_revision,
    'model_revision': model_revision,
    'method': 'PO/idk',
    'split': 'forget5',
    'num_processes': 1,
    'batch_size_per_process': 4,
    'gradient_accumulation_steps': 32,
    'effective_forget_batch': 128,
    'effective_total_sequences_per_update': 256,
    'epochs': 8,
    'expected_optimizer_updates_per_epoch': 4,
    'expected_optimizer_updates_total': 32,
    'precision': 'fp16',
    'tokenizer': 'fast',
    'deepspeed_zero_stage': 2,
    'optimizer_offload': 'cpu',
    'parameter_offload': 'cpu',
}
Path('/kaggle/working/po_single_t4_b4a32_fast_manifest.json').write_text(
    json.dumps(manifest, indent=2)
)
print(json.dumps(manifest, indent=2))
print('PRETRAIN_AUDIT_OK')


## 학습

다음 셀은 한 번만 실행하세요. Kaggle 화면에 GPU가 두 장 표시되어도 `CUDA_VISIBLE_DEVICES=0`과 `num_processes: 1` 때문에 학습에는 한 장만 참여합니다. 처음 optimizer step은 모델 로딩과 CPU offload 초기화 때문에 오래 걸릴 수 있습니다.


In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/FIUBench_kaggle
export HF_HOME=/kaggle/working/hf_cache
export TOKENIZERS_PARALLELISM=false
export HYDRA_FULL_ERROR=1
export PYTHONUNBUFFERED=1
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
CUDA_VISIBLE_DEVICES=0 DS_SKIP_CUDA_CHECK=1 accelerate launch \
  --config_file config/accelerate_kaggle_single_t4_po.yaml \
  --main_process_port 2234 \
  forget.py --config-name forget_po_forget5_single_t4_b4a32_fast \
  2>&1 | tee /kaggle/working/po_forget5_single_t4_b4a32_fast_console.log
echo 'TRAIN_PROCESS_EXIT_0'


In [ ]:
from pathlib import Path
import re, torch

out = Path('/kaggle/working/models/llava_phi_PO_forget5_single_t4_b4a32_fast')
checkpoint = out / 'checkpoint.pt'
log_path = Path('/kaggle/working/po_forget5_single_t4_b4a32_fast_console.log')
assert checkpoint.is_file(), '최종 checkpoint가 없습니다. 콘솔 로그를 확인하세요.'
assert log_path.is_file()
log_text = log_path.read_text(errors='replace')
assert 'Num processes: 1' in log_text or 'num_processes' not in log_text
assert 'Total train batch size (w. parallel, distributed & accumulation) = 128' in log_text

state = torch.load(checkpoint, map_location='cpu', weights_only=True)
assert state
assert all(torch.isfinite(t).all().item() for t in state.values())
intermediate = sorted(out.glob('step_*/checkpoint.pt'))
print('final checkpoint:', checkpoint)
print('checkpoint MiB:', checkpoint.stat().st_size / 1024**2)
print('saved tensors:', len(state))
print('intermediate checkpoints:', [str(p) for p in intermediate])
print('console log:', log_path)
del state
print('SINGLE_T4_PO_B4_A32_TRAINING_AND_CHECKPOINT_AUDIT_OK')


## 학습 후

마지막 성공 문구와 출력 경로를 확인한 뒤 **Save Version**으로 Output을 보존하세요. 이후 `fast_eval` 노트북에는 이 저장 Version Output만 Input으로 연결하고 `METHOD = "PO"`로 평가하면 됩니다. 다른 PO/GA Output은 함께 연결하지 않는 편이 안전합니다.

함께 보존할 파일:

- `models/llava_phi_PO_forget5_single_t4_b4a32_fast/`
- `po_forget5_single_t4_b4a32_fast_console.log`
- `po_single_t4_b4a32_fast_manifest.json`
- `fiu_environment_single_t4_b4a32_fast.txt`
- `fiu_runtime_single_t4_b4a32_fast.patch`
- `fiu_data_revision.txt`
